# Modulo 8 · Lezione 8 — RNN e Transformer: Sequenze e NLP Moderno

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Il problema delle sequenze |
| 2 | RNN semplice: memoria a breve termine |
| 3 | LSTM e GRU: memoria a lungo termine |
| 4 | Embedding: rappresentare il testo |
| 5 | Transformer e attenzione: lo stato dell'arte |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

---
## 1. Il problema delle sequenze

```
I dati che abbiamo visto finora sono 'statici':
  immagine → classe, feature → prezzo, ...

Molti problemi hanno una struttura TEMPORALE o SEQUENZIALE:
  Testo:              "Il gatto mangia" → l'ordine conta
  Serie temporali:    temperatura, vendite, borsa
  Audio:               segnale sonoro, speech
  DNA:                 sequenza di nucleotidi
  Codice:               token di un programma

L'ordine conta:
  "cane morde uomo" ≠ "uomo morde cane"

Reti dense: trattano tutti gli input come indipendenti
  → perdono tutta l'informazione temporale/posizionale

RNN: elaborano la sequenza passo dopo passo,
     mantenendo uno 'stato nascosto' che funge da memoria
```

In [ ]:
# Tipi di compiti su sequenze
print("""
Tipi di compiti con sequenze:

  One-to-one:    ingresso fisso  → uscita fissa       (classificazione di immagini)
  One-to-many:   ingresso fisso  → sequenza            (generazione di testo da immagine)
  Many-to-one:   sequenza        → uscita fissa        (analisi del sentimento)
  Many-to-many:  sequenza        → sequenza (stessa)   (etichettatura POS)
  Many-to-many:  sequenza        → sequenza (diversa)  (traduzione automatica)

Architetture:
  RNN (Recurrent Neural Network)  → memoria di base
  LSTM (Long Short-Term Memory)   → memoria a lungo termine
  GRU (Gated Recurrent Unit)      → LSTM semplificata, più veloce
  Transformer                     → attenzione senza ricorrenza (stato dell'arte)
""")

---
## 2. RNN semplice: memoria a breve termine

In [ ]:
# Implementare una RNN semplice manualmente
class RNNSimple:
    """
    Per ogni passo t:
      h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)

    h_t: stato nascosto (la 'memoria')
    x_t: ingresso al passo t
    """
    def __init__(self, n_entrada, n_oculta):
        # Inizializzazione Glorot
        lim = np.sqrt(6 / (n_entrada + n_oculta))
        self.Wx = np.random.uniform(-lim, lim, (n_entrada, n_oculta))
        self.Wh = np.random.uniform(-lim, lim, (n_oculta,  n_oculta))
        self.b  = np.zeros(n_oculta)
        self.n_oculta = n_oculta

    def forward(self, secuencia):
        """secuencia: (T, n_entrada) → restituisce tutti gli stati nascosti"""
        T = len(secuencia)
        h = np.zeros(self.n_oculta)  # stato iniziale
        estados = []
        for t in range(T):
            h = np.tanh(secuencia[t] @ self.Wx + h @ self.Wh + self.b)
            estados.append(h.copy())
        return np.array(estados), h   # tutti gli stati, ultimo stato

# Demo: elaborare una sequenza di 5 passi
rnn = RNNSimple(n_entrada=3, n_oculta=4)
secuencia = np.random.randn(5, 3)  # 5 passi, 3 feature ciascuno

estados, h_final = rnn.forward(secuencia)
print("RNN semplice:")
print(f"  Sequenza: {secuencia.shape}  (T, n_entrada)")
print(f"  Stati:    {estados.shape}    (T, n_oculta)")
print(f"  h_final:  {h_final.round(4)}  ← riassume tutta la sequenza")

# Visualizzare gli stati nascosti
fig, ax = plt.subplots(figsize=(9, 4))
for i in range(4):
    ax.plot(estados[:, i], 'o-', lw=2, label=f'h[{i}]')
ax.set_xlabel('Passo temporale'); ax.set_ylabel('Attivazione')
ax.set_title('Evoluzione dello stato nascosto h_t in una RNN semplice')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Il problema della RNN semplice: gradiente evanescente su sequenze lunghe
print("Problema: gradiente evanescente su sequenze lunghe")
print("="*55)

longitudes = [1, 5, 10, 20, 50, 100]
# La derivata di tanh(z) ha massimo 1 in z=0, ma tipicamente ~0.5
# Con una sequenza di T passi, il gradiente si moltiplica T volte per ~0.5
factor = 0.5
for T in longitudes:
    grad = factor ** T
    print(f"  T={T:3d} passi: gradiente ≈ 0.5^{T} = {grad:.2e}  {'← praticamente 0' if grad < 1e-6 else ''}")

print("\n→ La RNN 'dimentica' ciò che è successo oltre 10-20 passi prima")
print("→ Soluzione: LSTM e GRU con porte (gate) che controllano il flusso di informazione")

---
## 3. LSTM e GRU: memoria a lungo termine

```
LSTM (Hochreiter & Schmidhuber, 1997):
  Aggiunge una 'cella di stato' (c_t) che può:
    - Ricordare: porta di ingresso (input gate)
    - Dimenticare: porta di oblio (forget gate)
    - Leggere:    porta di uscita (output gate)

  f_t = sigmoid(Wf·[h_{t-1}, x_t] + bf)  ← forget gate: quanto dimenticare?
  i_t = sigmoid(Wi·[h_{t-1}, x_t] + bi)  ← input gate:  cosa salvare di nuovo?
  g_t = tanh(Wg·[h_{t-1}, x_t] + bg)     ← candidato nuovo contenuto
  c_t = f_t * c_{t-1} + i_t * g_t        ← aggiornare la cella di stato
  o_t = sigmoid(Wo·[h_{t-1}, x_t] + bo)  ← output gate
  h_t = o_t * tanh(c_t)                   ← stato nascosto

GRU (Cho et al., 2014):
  Versione semplificata con solo 2 porte:
    - Porta di aggiornamento (update gate)
    - Porta di reset (reset gate)
  Meno parametri → più veloce, quasi altrettanto efficace

In Keras:
  layers.LSTM(units, return_sequences=False)
  layers.GRU(units, return_sequences=False)

  return_sequences=True  → restituisce (T, units) — per impilare RNN
  return_sequences=False → restituisce (units,)   — solo l'ultimo stato
```

In [ ]:
# Demo: strati ricorrenti in Keras
# Sequenza di 10 passi, ciascuno con 8 feature
x_seq = keras.Input(shape=(10, 8))  # (batch, T, features)

# SimpleRNN
h_rnn  = layers.SimpleRNN(32)(x_seq)
# LSTM
h_lstm = layers.LSTM(32)(x_seq)
# GRU
h_gru  = layers.GRU(32)(x_seq)

print("Uscite con return_sequences=False (solo ultimo stato):")
for nombre, h in [('SimpleRNN', h_rnn), ('LSTM', h_lstm), ('GRU', h_gru)]:
    print(f"  {nombre}: {h.shape}")

# Con return_sequences=True
h_seq = layers.LSTM(32, return_sequences=True)(x_seq)
print(f"\nLSTM con return_sequences=True: {h_seq.shape}  (batch, T, units)")

# Impilare LSTM
h1 = layers.LSTM(64, return_sequences=True)(x_seq)   # restituisce sequenza
h2 = layers.LSTM(32)(h1)                              # consuma sequenza, restituisce vettore
print(f"\nLSTM impilate: {x_seq.shape} → {h1.shape} → {h2.shape}")

# Parametri di ogni tipo
for nombre, cap in [('SimpleRNN(32)', layers.SimpleRNN(32)),
                     ('GRU(32)',       layers.GRU(32)),
                     ('LSTM(32)',      layers.LSTM(32))]:
    m = keras.Sequential([keras.Input(shape=(10,8)), cap])
    print(f"  {nombre}: {m.count_params():,} parametri")

In [ ]:
# Applicazione: previsione di serie temporali (temperatura sintetica)
# Generare la serie: temperatura con tendenza stagionale
T = 500
t = np.linspace(0, 10 * np.pi, T)
temperatura = np.sin(t) + 0.5*np.sin(3*t) + 0.3*np.random.randn(T)

# Creare finestre di ingresso/uscita
N_PASOS = 20  # predire con i 20 passi precedenti
X_ts = np.array([temperatura[i:i+N_PASOS] for i in range(len(temperatura)-N_PASOS-1)])
y_ts = temperatura[N_PASOS+1:]

X_ts = X_ts[..., np.newaxis]  # aggiungere la dimensione feature: (N, 20, 1)

split = int(0.8 * len(X_ts))
X_tr_ts, X_te_ts = X_ts[:split], X_ts[split:]
y_tr_ts, y_te_ts = y_ts[:split], y_ts[split:]

print(f"Serie temporale:")
print(f"  Campioni train: {X_tr_ts.shape}")
print(f"  Campioni test:  {X_te_ts.shape}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(temperatura, lw=1, color='steelblue')
ax.axvline(split + N_PASOS, color='red', ls='--', label='train/test split')
ax.set_title('Serie temporale di temperatura sintetica')
ax.set_xlabel('Tempo'); ax.set_ylabel('Temperatura'); ax.legend()
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Confrontare: Dense vs LSTM per la previsione di serie temporali
tf.random.set_seed(42)

# LSTM
m_lstm = keras.Sequential([
    layers.Input(shape=(N_PASOS, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
], name='LSTM_ts')
m_lstm.compile('adam', 'mse', ['mae'])
h_lstm = m_lstm.fit(X_tr_ts, y_tr_ts, epochs=30, batch_size=32,
                    validation_split=0.1, verbose=0)
_, mae_lstm = m_lstm.evaluate(X_te_ts, y_te_ts, verbose=0)

# Dense (senza memoria temporale)
m_dense_ts = keras.Sequential([
    layers.Input(shape=(N_PASOS, 1)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
], name='Dense_ts')
m_dense_ts.compile('adam', 'mse', ['mae'])
m_dense_ts.fit(X_tr_ts, y_tr_ts, epochs=30, batch_size=32,
               validation_split=0.1, verbose=0)
_, mae_dense = m_dense_ts.evaluate(X_te_ts, y_te_ts, verbose=0)

print(f"Dense → MAE: {mae_dense:.4f}")
print(f"LSTM  → MAE: {mae_lstm:.4f}")

# Visualizzare le predizioni
y_pred_lstm  = m_lstm.predict(X_te_ts, verbose=0).ravel()
y_pred_dense = m_dense_ts.predict(X_te_ts, verbose=0).ravel()

fig, ax = plt.subplots(figsize=(12, 4))
x_plot = range(len(y_te_ts))
ax.plot(x_plot, y_te_ts,     'k-',        lw=1.5, label='Reale', alpha=0.7)
ax.plot(x_plot, y_pred_lstm, 'steelblue', lw=1.5, label=f'LSTM (MAE={mae_lstm:.3f})')
ax.plot(x_plot, y_pred_dense,'coral',     lw=1.5, label=f'Dense (MAE={mae_dense:.3f})', alpha=0.7)
ax.set_title('Previsione di serie temporale: Dense vs LSTM')
ax.set_xlabel('Tempo'); ax.set_ylabel('Temperatura')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Embedding: rappresentare il testo

In [ ]:
print("""
Come convertire il testo in numeri?

  One-hot: vocabolario di 10000 parole → vettore di 10000 zeri con un 1
    Problema: 10000 dimensioni, senza somiglianza semantica
    "re" e "regina" sono distanti quanto "re" e "sedia"

  Embedding: mappare ogni parola in un vettore denso di 128 dimensioni
    I vettori si imparano durante l'addestramento
    Parole simili risultano vicine nello spazio:
      vec(re) - vec(uomo) + vec(donna) ≈ vec(regina)  [Word2Vec]

layers.Embedding(vocab_size, embed_dim):
  vocab_size: numero di token distinti
  embed_dim:  dimensione del vettore (tipico: 64-512)
  Input: (batch, T) → interi (id dei token)
  Output: (batch, T, embed_dim)
""")

# Dimostrazione: strato Embedding
VOCAB = 100
EMBED = 16
SEQ_LEN = 10

embedding = layers.Embedding(VOCAB, EMBED)
# Simulare una sequenza di token (interi 0-99)
tokens = tf.constant([[5, 12, 3, 45, 0, 7, 89, 22, 4, 1]])
vectors = embedding(tokens)
print(f"Input token: {tokens.shape}  (batch, seq_len)")
print(f"Output embedding: {vectors.shape}  (batch, seq_len, embed_dim)")
print(f"\nVettore del token 5: {vectors[0,0,:].numpy().round(3)}")

In [ ]:
# Analisi del sentimento con IMDB: recensioni cinematografiche → positivo/negativo
MAX_FEATURES = 10000  # vocabolario: le 10000 parole più frequenti
MAX_LEN      = 200    # tagliare/riempire tutte le recensioni a 200 token

(X_tr_imdb, y_tr_imdb), (X_te_imdb, y_te_imdb) = keras.datasets.imdb.load_data(
    num_words=MAX_FEATURES
)

print(f"Train: {len(X_tr_imdb)} recensioni")
print(f"Esempio — token: {X_tr_imdb[0][:15]}...")
print(f"Etichetta: {y_tr_imdb[0]} (1=positivo, 0=negativo)")

# Pad/troncare a MAX_LEN
X_tr_imdb = keras.utils.pad_sequences(X_tr_imdb, maxlen=MAX_LEN)
X_te_imdb = keras.utils.pad_sequences(X_te_imdb, maxlen=MAX_LEN)
print(f"\nDopo il pad: {X_tr_imdb.shape}")

In [ ]:
# Modello: Embedding + LSTM + classificatore
tf.random.set_seed(42)
modelo_nlp = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(MAX_FEATURES, 64),          # (N, 200) → (N, 200, 64)
    layers.LSTM(64, return_sequences=True),       # (N, 200, 64) → (N, 200, 64)
    layers.LSTM(32),                              # (N, 200, 64) → (N, 32)
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),        # binario: pos/neg
], name='LSTM_sentiment')

modelo_nlp.summary()

modelo_nlp.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

h_nlp = modelo_nlp.fit(
    X_tr_imdb, y_tr_imdb,
    epochs=5, batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, acc_nlp = modelo_nlp.evaluate(X_te_imdb, y_te_imdb, verbose=0)
print(f"\nTest accuracy (IMDB): {acc_nlp:.4f} ({acc_nlp:.2%})")

---
## 5. Transformer: lo stato dell'arte

```
Problema delle RNN:
  1. Elaborazione sequenziale → lenta su GPU
  2. Anche con LSTM, le sequenze molto lunghe sono difficili
  3. Per tradurre "the animal didn't cross the street because it was tired"
     "it" si riferisce ad "animal" o a "street"? Molto lontano nella sequenza.

Transformer (Vaswani et al., "Attention Is All You Need", 2017):
  - Elabora TUTTI i token IN PARALLELO (non sequenzialmente)
  - Meccanismo di ATTENZIONE: ogni token può 'prestare attenzione' a qualsiasi altro
  - Permette relazioni a qualsiasi distanza con lo stesso costo

Attenzione (Attention):
  Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V

  Q (Query):  cosa cerca questo token?
  K (Key):    cosa offre ogni token?
  V (Value):  quale informazione contiene ogni token?

  Il softmax converte le similarità in pesi di attenzione.
  Ogni token pesa l'informazione di TUTTI i token in base alla rilevanza.

Modelli basati su Transformer:
  BERT (Google, 2018)    → comprensione del testo
  GPT-2/3/4 (OpenAI)    → generazione di testo
  T5, LLaMA, Mistral    → multiuso
  ViT (Vision Transformer) → immagini
```

In [ ]:
# Visualizzare come funziona l'attenzione
np.random.seed(42)

frase = ['Il', 'gatto', 'non', 'ha attraversato', 'la', 'strada', 'perché', 'era', 'stanco']
T = len(frase)

# Simulare i pesi di attenzione per il token 'era' (idx 7)
# In realtà si imparano; qui li creiamo a mano per illustrare
pesos_atencion = np.array([
    [0.05, 0.35, 0.02, 0.03, 0.02, 0.03, 0.05, 0.15, 0.30],  # 'era' → presta attenzione a 'gatto' e 'stanco'
    [0.20, 0.05, 0.03, 0.15, 0.02, 0.03, 0.05, 0.25, 0.22],  # 'attraversato' → presta attenzione a 'gatto' e 'era'
    [0.05, 0.05, 0.05, 0.05, 0.05, 0.40, 0.05, 0.15, 0.15],  # 'stanco' → presta attenzione a 'strada'
])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
tokens_consultados = ['era', 'attraversato', 'stanco']
idx_consultados = [7, 3, 8]

for ax, (token_q, pesos) in zip(axes, zip(tokens_consultados, pesos_atencion)):
    bars = ax.barh(range(T), pesos, color=['red' if i == idx_consultados[tokens_consultados.index(token_q)]
                   else 'steelblue' for i in range(T)], alpha=0.8)
    ax.set_yticks(range(T)); ax.set_yticklabels(frase, fontsize=11)
    ax.set_xlabel('Peso di attenzione')
    ax.set_title(f'Il token "{token_q}" presta attenzione a:')
    ax.grid(True, alpha=0.3, axis='x')
    for bar, peso in zip(bars, pesos):
        ax.text(peso + 0.005, bar.get_y() + bar.get_height()/2,
                f'{peso:.2f}', va='center', fontsize=8)

plt.suptitle('Meccanismo di attenzione (simulato)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Multi-Head Attention in Keras
print("Multi-Head Attention in Keras:")
x = keras.Input(shape=(MAX_LEN, 64))  # (batch, seq, embed)
att = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
print(f"  Input:  {x.shape}")
print(f"  Output: {att.shape}")

# Blocco Transformer base per la classificazione del sentimento
print("\nCostruttore di un blocco Transformer:")
print("""
def transformer_block(x, embed_dim, num_heads, ff_dim, dropout=0.1):
    # Multi-Head Self-Attention
    attn = layers.MultiHeadAttention(num_heads, key_dim=embed_dim//num_heads)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x    = layers.LayerNormalization()(x + attn)   # residuo + norm

    # Feed-Forward Network
    ff  = layers.Dense(ff_dim, activation='relu')(x)
    ff  = layers.Dense(embed_dim)(ff)
    ff  = layers.Dropout(dropout)(ff)
    out = layers.LayerNormalization()(x + ff)       # residuo + norm
    return out
""")

def transformer_block(x, embed_dim, num_heads, ff_dim, dropout=0.1):
    attn = layers.MultiHeadAttention(num_heads, key_dim=embed_dim//num_heads)(x, x)
    attn = layers.Dropout(dropout)(attn)
    x    = layers.LayerNormalization()(x + attn)
    ff   = layers.Dense(ff_dim, activation='relu')(x)
    ff   = layers.Dense(embed_dim)(ff)
    ff   = layers.Dropout(dropout)(ff)
    return layers.LayerNormalization()(x + ff)

# Modello Transformer per IMDB
tf.random.set_seed(42)
EMBED_DIM = 64

entradas = keras.Input(shape=(MAX_LEN,))
x = layers.Embedding(MAX_FEATURES, EMBED_DIM)(entradas)
x = transformer_block(x, EMBED_DIM, num_heads=4, ff_dim=128)
x = layers.GlobalAveragePooling1D()(x)  # riassumere l'intera sequenza
x = layers.Dropout(0.3)(x)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dropout(0.2)(x)
salidas = layers.Dense(1, activation='sigmoid')(x)

modelo_transformer = keras.Model(entradas, salidas, name='Transformer_IMDB')
modelo_transformer.compile('adam', 'binary_crossentropy', ['accuracy'])
modelo_transformer.summary()

h_tr = modelo_transformer.fit(
    X_tr_imdb, y_tr_imdb,
    epochs=5, batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, acc_tr = modelo_transformer.evaluate(X_te_imdb, y_te_imdb, verbose=0)
print(f"\nTransformer — Test accuracy: {acc_tr:.4f}")
print(f"LSTM        — Test accuracy: {acc_nlp:.4f}")

---
## Pratica

### Esercizio 1 — GRU per IMDB

**Consegna:** Le GRU sono più veloci delle LSTM.

1. Costruisci lo stesso modello di analisi del sentimento ma con GRU invece di LSTM
2. Confronta il tempo di addestramento e l'accuracy con il modello LSTM
3. Vale la pena la complessità extra della LSTM?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

MAX_FEATURES, MAX_LEN = 10000, 200
(X_tr, y_tr), (X_te, y_te) = keras.datasets.imdb.load_data(num_words=MAX_FEATURES)
X_tr = keras.utils.pad_sequences(X_tr, maxlen=MAX_LEN)
X_te = keras.utils.pad_sequences(X_te, maxlen=MAX_LEN)

def build_rnn(tipo='lstm'):
    rnn_capa = layers.LSTM(64, return_sequences=True) if tipo == 'lstm' else layers.GRU(64, return_sequences=True)
    rnn_capa2 = layers.LSTM(32) if tipo == 'lstm' else layers.GRU(32)
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(MAX_FEATURES, 64),
        rnn_capa, rnn_capa2,
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ])

resultados = {}
for tipo in ['lstm', 'gru']:
    tf.random.set_seed(42)
    m = build_rnn(tipo)
    m.compile('adam', 'binary_crossentropy', ['accuracy'])
    t0 = time.time()
    m.fit(X_tr, y_tr, epochs=3, batch_size=128, validation_split=0.1, verbose=0)
    t1 = time.time()
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    resultados[tipo] = {'acc': acc, 'tiempo': t1-t0, 'params': m.count_params()}
    print(f"{tipo.upper()}: acc={acc:.4f}, tempo={t1-t0:.1f}s, params={m.count_params():,}")

print(f"\nGRU è {resultados['lstm']['tiempo']/resultados['gru']['tiempo']:.1f}x più veloce")
print(f"Differenza in accuracy: {abs(resultados['lstm']['acc']-resultados['gru']['acc']):.4f}")
```
</details>

---
### Esercizio 2 — Previsione multivariata

**Consegna:** La serie temporale precedente aveva solo 1 feature. Prevedi usando più variabili:

1. Genera: temperatura + umidità (correlate + rumore)
2. Usa finestre di 20 passi con 2 feature → prevedi la temperatura
3. Confronta: LSTM con 1 feature vs LSTM con 2 feature
4. Aiuta conoscere l'umidità?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

T = 600
t = np.linspace(0, 12*np.pi, T)
temp     = np.sin(t) + 0.5*np.sin(3*t) + 0.3*np.random.randn(T)
humedad  = -0.6*np.sin(t) + 0.3*np.random.randn(T) + 0.5  # correlata inversamente

# Normalizzare
temp_n    = (temp - temp.mean()) / temp.std()
humedad_n = (humedad - humedad.mean()) / humedad.std()

N_PASOS = 20
def crear_ventanas(features, target, n):
    X = np.array([features[i:i+n] for i in range(len(target)-n-1)])
    y = target[n+1:]
    return X, y

# 1 feature
X1 = temp_n[:, None]
X_ts1, y_ts = crear_ventanas(X1, temp_n, N_PASOS)

# 2 feature
X2 = np.stack([temp_n, humedad_n], axis=-1)
X_ts2, _   = crear_ventanas(X2, temp_n, N_PASOS)

split = int(0.8 * len(y_ts))

maes = {}
for nombre, X_ts in [('1 feature (temp)', X_ts1), ('2 feature (temp+umidità)', X_ts2)]:
    tf.random.set_seed(42)
    n_feat = X_ts.shape[-1]
    m = keras.Sequential([
        layers.Input(shape=(N_PASOS, n_feat)),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(1),
    ])
    m.compile('adam', 'mse', ['mae'])
    m.fit(X_ts[:split], y_ts[:split], epochs=25, batch_size=32,
          validation_split=0.1, verbose=0)
    _, mae = m.evaluate(X_ts[split:], y_ts[split:], verbose=0)
    maes[nombre] = mae
    print(f"{nombre}: MAE={mae:.4f}")

print(f"\nMiglioramento: {maes['1 feature (temp)'] - maes['2 feature (temp+umidità)']:.4f}")
```
</details>

---
### Esercizio 3 — Embedding personalizzato

**Consegna:** Visualizza cosa imparano gli embedding.

1. Addestra il modello LSTM di IMDB (già addestrato)
2. Estrai i pesi dello strato Embedding: `modelo.layers[1].get_weights()[0]`
3. Usa PCA per ridurre a 2D
4. Visualizza le 20 parole più frequenti nello spazio 2D
5. Le parole simili risultano vicine?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Dizionario inverso IMDB
word_index = keras.datasets.imdb.get_word_index()
idx2word   = {v+3: k for k, v in word_index.items()}
idx2word.update({0:'<PAD>', 1:'<START>', 2:'<UNK>', 3:'<UNUSED>'})

# Estrarre gli embedding
embed_pesos = modelo_nlp.layers[1].get_weights()[0]  # (10000, 64)
print(f"Pesi dell'Embedding: {embed_pesos.shape}")

# Le parole più frequenti sono quelle con indice più basso (IMDB le ordina per frequenza)
n_show = 40
indices = list(range(4, 4 + n_show))
vectores = embed_pesos[indices]  # (40, 64)
palabras = [idx2word.get(i, '?') for i in indices]

# PCA a 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(vectores)  # (40, 2)

fig, ax = plt.subplots(figsize=(12, 9))
ax.scatter(coords[:,0], coords[:,1], c='steelblue', s=60, zorder=3)
for (x, y), palabra in zip(coords, palabras):
    ax.annotate(palabra, (x, y), textcoords='offset points',
                xytext=(5, 5), fontsize=9)
ax.set_title('Embedding di IMDB — PCA 2D (40 parole più frequenti)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Varianza spiegata da PC1+PC2: {pca.explained_variance_ratio_.sum():.2%}")
```
</details>

---
### Esercizio 4 — LSTM bidirezionale

**Consegna:** Una LSTM legge la sequenza da sinistra a destra. Una LSTM bidirezionale la legge in entrambe le direzioni e concatena i risultati.

```python
layers.Bidirectional(layers.LSTM(64))
```

1. Adatta il modello di analisi del sentimento per usare LSTM bidirezionale
2. Migliora l'accuracy su IMDB?
3. Quanti parametri extra ha?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

MAX_FEATURES, MAX_LEN = 10000, 200
(X_tr, y_tr), (X_te, y_te) = keras.datasets.imdb.load_data(num_words=MAX_FEATURES)
X_tr = keras.utils.pad_sequences(X_tr, maxlen=MAX_LEN)
X_te = keras.utils.pad_sequences(X_te, maxlen=MAX_LEN)

modelos = {
    'LSTM unidirezionale': keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(MAX_FEATURES, 64),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ]),
    'LSTM bidirezionale': keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(MAX_FEATURES, 64),
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        layers.Bidirectional(layers.LSTM(32)),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ]),
}

print(f"{'Modello':25} {'Params':>12} {'Test acc':>10}")
print('-' * 52)
for nombre, m in modelos.items():
    tf.random.set_seed(42)
    m.compile('adam', 'binary_crossentropy', ['accuracy'])
    m.fit(X_tr, y_tr, epochs=3, batch_size=128, validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_te, y_te, verbose=0)
    print(f"{nombre:25} {m.count_params():>12,} {acc:>10.4f}")
```
</details>

---
### Esercizio 5 (sfida) — Generazione di testo

**Consegna:** Addestra un modello per generare testo carattere per carattere.

1. Usa un testo breve (poesia, canzone, codice — minimo 1000 caratteri)
2. Codifica carattere → intero
3. Modello: Embedding(vocab) → LSTM(128) → Dense(vocab, softmax)
4. Addestra per predire il carattere successivo
5. Genera testo nuovo con campionamento (non argmax — usa temperatura=0.8)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# Testo di esempio (frammento di "In principio" — Genesi semplificata)
texto = """
in principio dio creo i cieli e la terra la terra era informe
e vuota e le tenebre coprivano la faccia dell abisso e lo spirito di dio
aleggiava sulla superficie delle acque e dio disse sia la luce e la luce fu e dio
vide che la luce era buona e dio separo la luce dalle tenebre e chiamo la
luce giorno e le tenebre chiamo notte e fu sera e fu mattina il primo giorno
e dio disse ci sia un firmamento in mezzo alle acque e separi le acque dalle acque
e dio fece il firmamento e separo le acque che stavano sotto il firmamento
dalle acque che stavano sopra il firmamento e cosi fu e dio chiamo il firmamento
cielo e fu sera e fu mattina il secondo giorno e dio disse si radunino le acque
che stanno sotto il cielo in un unico luogo e appaia l asciutto e cosi fu e dio
chiamo l asciutto terra e la raccolta delle acque chiamo mari e dio vide che era cosa buona
""" * 5

# Vocabolario di caratteri
chars  = sorted(set(texto))
vocab  = len(chars)
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
encoded = np.array([c2i[c] for c in texto])

print(f"Testo: {len(texto)} caratteri, {vocab} unici")

# Creare le sequenze
SEQ_LEN = 40
X = np.array([encoded[i:i+SEQ_LEN] for i in range(len(encoded)-SEQ_LEN-1)])
y = encoded[SEQ_LEN+1:]

# Modello
tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Embedding(vocab, 32),
    layers.LSTM(128, return_sequences=True),
    layers.LSTM(64),
    layers.Dense(vocab, activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy')
m.fit(X, y, epochs=30, batch_size=128, verbose=0)
print("Addestramento completato")

def generar(modelo, semilla, n_chars=200, temperatura=0.8):
    semilla_encoded = np.array([c2i.get(c, 0) for c in semilla[-SEQ_LEN:]])
    resultado = list(semilla)
    for _ in range(n_chars):
        x = semilla_encoded[-SEQ_LEN:][None, :]  # (1, SEQ_LEN)
        probs = modelo.predict(x, verbose=0)[0]   # (vocab,)
        probs = np.log(probs + 1e-8) / temperatura
        probs = np.exp(probs - probs.max())
        probs /= probs.sum()
        siguiente = np.random.choice(vocab, p=probs)
        resultado.append(i2c[siguiente])
        semilla_encoded = np.append(semilla_encoded, siguiente)
    return ''.join(resultado)

print("\n=== Testo generato ===")
print(generar(m, semilla='in principio ', n_chars=300, temperatura=0.8))
```
</details>

---
## Riepilogo

```python
# RNN per sequenze
modelo = keras.Sequential([
    layers.Input(shape=(T, n_features)),  # (batch, passi, feature)
    layers.LSTM(64, return_sequences=True),  # impilata → True
    layers.LSTM(32),                          # ultima → False
    layers.Dense(n_salidas),
])

# NLP: testo → embedding → RNN
modelo = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),         # (batch, T) interi
    layers.Embedding(VOCAB, embed_dim),     # (batch, T, embed_dim)
    layers.LSTM(64),                        # o GRU, o Bidirectional(LSTM())
    layers.Dense(1, activation='sigmoid'),  # o softmax per il multiclasse
])

# Blocco Transformer
layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
layers.GlobalAveragePooling1D()  # riassumere la sequenza in un vettore

# Tipi di strati ricorrenti
# SimpleRNN(n)          → base, dimentica rapidamente
# LSTM(n)               → memoria lunga, più parametri
# GRU(n)                → equilibrio velocità/prestazioni
# Bidirectional(LSTM(n)) → legge la sequenza in entrambe le direzioni
```

## Cosa viene nella prossima lezione?
Nella **Lezione 9** impareremo a **monitorare l'addestramento**: TensorBoard, learning rate schedule e tecniche avanzate di debugging dei modelli.